# Train

Chỉ có bốn phương pháp. Chọn cấu hình bằng validation, ghi snapshot trước khi Evaluate đọc nhãn test.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

project_directory = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                         if (path / "recommendation/config.json").is_file())
sys.path.insert(0, str(project_directory / "recommendation/src"))
from pipeline_common import ROOT, ARTIFACTS, REPORTS, read_json, load_csv


In [2]:
from train import fit_and_select
selection = fit_and_select()
display(pd.DataFrame({"configuration":selection["configurations"], "policy":selection["policies"]}))
print("Mặc định:",selection["winner"],"| alpha:",selection["alpha"])


,configuration,policy
Popularity,Popularity,coverage90
ContentBased,ContentFull,quality_only
CollaborativeFiltering,LogisticMF_d4_r5,coverage90
Hybrid,Hybrid_alpha_0.5,coverage90


Mặc định: Popularity | alpha: 0.5


## Validation và hội tụ

CF và content mỗi nhóm xuất một cấu hình tốt nhất trên validation; hybrid kết hợp hai cấu hình đó. Bảng tìm cấu hình chi tiết nằm trong artifacts, không thêm hàng phương pháp vào báo cáo chính.

In [3]:
display(load_csv(REPORTS / "validation_primary.csv"))
display(pd.DataFrame(read_json(ARTIFACTS / "models/primary/fit_audit.json")))


,model,precision_at_10,recall_at_10,f1_at_10,ndcg_at_10,coverage,concentration,observed_auc
0,CollaborativeFiltering,0.128892,0.322166,0.166904,0.238029,0.747049,0.552000,0.531349
1,ContentBased,0.135530,0.346253,0.176231,0.224750,0.933324,0.273116,0.509749
2,Hybrid,0.132605,0.341797,0.172147,0.244736,0.879406,0.462176,0.521576
3,Popularity,0.138301,0.347829,0.177645,0.248462,0.653033,0.500270,0.490773


,cohort,model,converged,iterations,objective,termination,maximum_iterations,relative_function_tolerance
0,historical_assumed_single,LogisticMF_d4_r5,True,88,11798.356991,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
1,historical_assumed_single,LogisticMF_d8_r5,True,117,11778.338615,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
2,run=2ad86635f1a18e06,LogisticMF_d4_r5,True,47,2102.387543,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
3,run=2ad86635f1a18e06,LogisticMF_d8_r5,True,47,2101.883921,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
4,run=94f1d592a7787780,LogisticMF_d4_r5,True,48,1538.764137,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
5,run=94f1d592a7787780,LogisticMF_d8_r5,True,37,1538.667426,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
6,run=e1d06d3cf6587c44,LogisticMF_d4_r5,True,57,2721.288085,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001
7,run=e1d06d3cf6587c44,LogisticMF_d8_r5,True,59,2718.864798,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...,350,0.000001


## Kết luận

Đã chốt bốn phương pháp và policy từng nhóm. Không đổi cấu hình theo kết quả test. Giữ trạng thái test đã được xem trong nghiên cứu trước để tránh gọi lần chạy này là xác nhận độc lập.